In [20]:
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

In [21]:
# Calculate figure width height dynamically based on desired padding between subplots such that subplots retain equal aspect ratio when
# padded through fig.subplot_adjust

def create_square_subplots_fixed(nrows, ncols, w_sub=4, h_sub=4,
                                 wspace=0.3, hspace=0.2,
                                 margin_left=0.5, margin_right=0.5,
                                 margin_top=0.7, margin_bottom=0.7,
                                 sharex=False, sharey=False):
    """
    Create subplots with exact square size, including proper spacing and margins.
    """
    # Compute total figure width and height in inches including subplot spacing
    fig_width = ncols*w_sub + (ncols-1)*w_sub*wspace + margin_left + margin_right
    fig_height = nrows*h_sub + (nrows-1)*h_sub*hspace + margin_top + margin_bottom

    # Fractions for subplots_adjust
    left_frac   = margin_left / fig_width
    right_frac  = 1 - margin_right / fig_width
    bottom_frac = margin_bottom / fig_height
    top_frac    = 1 - margin_top / fig_height

    # Adjust spacing as fraction of subplot size (this is correct now)
    wspace_frac = wspace
    hspace_frac = hspace

    # Create figure and axes
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width, fig_height),
                             sharex=sharex, sharey=sharey)

    plt.subplots_adjust(
        left=left_frac,
        right=right_frac,
        bottom=bottom_frac,
        top=top_frac,
        wspace=wspace_frac,
        hspace=hspace_frac
    )

    return fig, axes


## Calibrated strain time series containing GW150914

In [3]:
nrt_strain_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt")

# -- Set a GPS time:
t0 = 1126259462.4    # -- GW150914
#-- Choose detector as H1, L1, or V1

strain = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_I/partial_successful_reconstruct_and_where_is_the_signal/store/GW150914_strain.pickle", absolute_path=True)
data_ar = 1e19 * strain.value

zero_time = 1.1262594e9 + 60 + 2.422 + 0.00109 # I got this zero time by looking at the caption of the figure produced by strain.plot().
time_nr_template = convert_gps_to_seconds(nrt_time_values, t0=zero_time, )
time_strain_data = convert_gps_to_seconds(strain.times, t0=zero_time)

# plt.figure(figsize=(8,4))
# plt.plot(time_strain_data, data_ar, "-", color="black", lw=1)
# thesis_plot(save_fig=False)

/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


In [4]:
_, axs = plt.subplots(nrows=2, ncols=1, sharex=False, figsize=(8,2+2))
axs[0].plot(time_nr_template, nrt_strain_values, color="blue")
axs[1].plot(time_nr_template, nrt_strain_values, color="blue")
axs[1].plot(time_strain_data, data_ar, color="black")

axs[0].set_xlim(-0.2, 0.036)
axs[1].set_xlim(-0.2, 0.036)
axs[0].set_ylim(-0.015, 0.015)

axs[0].set_ylabel(r"$h(t)$ $\:\mathrm{[10^{-19}]}$")
axs[1].set_ylabel(r"$d(t)$ $\:\mathrm{[10^{-19}]}$")
axs[1].set_xlabel(r"Time $t$ $\mathrm{[s]}$")

save_figure(False)

Matplotlib is building the font cache; this may take a moment.


## Welch averaged PSD

In [ ]:
# events = [
#     {'desired_duration': 32, 'unpack': True, 'gps_center': 1126259462.4,
#      'absolute_path':
#          '/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects_II/H1_GW150914.hdf5'
#      },
#     {
#     'desired_duration': 32, 'unpack': True, 'gps_center': 1242459857.4,
#      'absolute_path':
#          '/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects_II/L1_GW190521_074359'
#     },
# ]
#
# time_event_1, strain_event_1 = _get(**events[0])
# time_event_2, strain_event_2 = _get_strain_data(**events[1])

In [4]:
GW150914 = get_strain_from_disc()
GW190521_074359 = get_strain_from_disc("GW190521_074359", detector="L1")

time_event_1, strain_event_1 = GW150914.time, GW150914.strain
time_event_2, strain_event_2 = GW190521_074359.time, GW190521_074359.strain

In [16]:
import jax.numpy as jnp

from scipy.signal.windows import tukey, hann

tukey_taper = lambda d: tukey(M=len(d), alpha=0.1, sym=True)
hann_taper = lambda d: hann(M=len(d), sym=True)


# Build skeleton
_, axs = plt.subplots(nrows=4, ncols=1, sharex=True, sharey=True, figsize=(8,4*4))

axs = np.array(axs).reshape(2,2)

for ax in axs:
    ax[0].loglog()

axs[0][0].set_ylabel(r"Noise power")
axs[0][1].set_ylabel(r"Noise power")
axs[1][0].set_ylabel(r"Noise power")
axs[1][0].set_ylabel(r"Noise power")
axs[1][1].set_xlabel(r"Frequency $f$ $\mathrm{[Hz]}$")


# Fill: upper left plot
x_ul, y_ul, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=2, final_average_call=jnp.mean, tapering_function=tukey_taper)

y_diff = 1e-4
axs[0][0].plot(x_ul, y_ul, color="black", label=r"GW150914, Tukey-windowed, window size $2\:\mathrm{s}$")
axs[0][0].legend()
axs[0][0].annotate(
    "",
    xy=(25.8, y_diff*2.39e-05),
    xytext=(7.47, y_diff*0.464),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.1"
    )
)
axs[0][0].text(2.1, y_diff*1.2e-3, "Seismic wall")
axs[0][0].text(10, y_diff*2.59e-7, "Thermal noise")
axs[0][0].annotate(
    "",
    xy=(125.53, y_diff*9.838e-7),
    xytext=(28.6, y_diff*1.79e-5),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.2"
    )
)
axs[0][0].text(300, y_diff*5e-8, "Photon shot\n noise")
axs[0][0].annotate(
    "",
    xy=(1394, y_diff*8e-6),
    xytext=(162, y_diff*8e-7),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.05"
    )
)
axs[0][0].annotate(
    "Power grid",
    xy=(62.0173, y_diff*0.17634),
    xytext=(47.9154, y_diff*1296.76),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0."
    )
)
axs[0][0].annotate(
    "Violin modes",
    xy=(323, y_diff*0.145),
    xytext=(233, y_diff*1177),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0."
    )
)


# Fill: upper right plot
x_ur, y_ur, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=2, final_average_call=jnp.mean, tapering_function=hann_taper)

axs[0][1].plot(x_ul, y_ul, color="black", alpha=1)
axs[0][1].plot(x_ur, y_ur, label="GW150914, Hann-windowed", color=blue, alpha=0.5)
axs[0][1].legend(loc="lower left")


# Fill: Lower left plot
x_ll, y_ll, _ = calculate_welch_average(x=time_event_2, y=strain_event_2, L=2, final_average_call=jnp.mean, tapering_function=tukey_taper)
axs[1][0].plot(x_ul, y_ul, color="black", alpha=1)
axs[1][0].plot(x_ll, y_ll, label="GW190521_074359, Tukey-windowed", color=red, alpha=0.5)
axs[1][0].legend(loc="lower left")


# Fill: Lower right plot
x_lr, y_lr, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=1, final_average_call=jnp.mean, tapering_function=tukey_taper)
axs[1][1].plot(x_ul, y_ul, color="black", alpha=1)
axs[1][1].plot(x_lr, y_lr, label=r"GW150914, window size $1\:\mathrm{s}$", color=green, alpha=0.5)
axs[1][1].legend(loc="lower left")


# Label subplots to refer back to in Tex Doc

labels = ["(a)", "(b)", "(c)", "(d)"]

for ax, lab in zip(axs.flat, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
    )


plt.tight_layout()
save_figure(save_fig=True)


	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 4.542402182156302
	Compare with area under welch ps:  4.545029553571364
	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 1.8930158549142369
	Compare with area under welch ps:  1.8934345771079526
	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 5.5976986426560575
	Compare with area under welch ps:  5.600608426977171
	Constructing 31 windows over which we average.
	Mean variance of tapered windows: 4.553104257495355
	Compare with area under welch ps:  4.5593994948333885


## Tukey vs. Hann window

In [ ]:
import jax.numpy as jnp
from scipy.signal.windows import hann
from scipy.signal.windows import tukey

tukey_taper = lambda d: tukey(M=len(d), alpha=0.1, sym=True)
hann_taper = lambda d: hann(M=len(d), sym=True)


x = np.linspace(-1, 1, 500)
y = np.ones(500)

y_tukey = tukey_taper(y) * y
y_hann = hann_taper(y) * y

_ = plt.figure(figsize=(8,4))
plt.plot(x, y_tukey, label="Tukey window", color="black")
plt.plot(x, y_hann, label="Hann window", color=blue,)
plt.legend(loc="best")
thesis_plot(xl="$x$", yl="$y$", save_fig=False, mode="longer")



## Introduction to the Wigner function

In [22]:
# Produce the data
n_pix = 2000
time = np.linspace(0, 1, n_pix)
dt = time[1] - time[0]

dirac_delta_time_norm = 1/np.sqrt(dt)

# Upper left
Xi_ul = np.random.standard_normal((n_pix,n_pix))

# Upper right
xi = np.random.standard_normal(n_pix) * dirac_delta_time_norm
S, t, f = Stress_jft(xi=xi, time=time, supress_print=True)

# Properly norm upper left
df = f[1]-f[0]
dirac_delta_freq_norm = 1/np.sqrt(df)
Xi_ul *= dirac_delta_freq_norm * dirac_delta_time_norm

In [44]:
fig, axs = create_square_subplots_fixed(sharex=True, sharey=True,
                                        nrows=2, ncols=2, w_sub=4, h_sub=4,
                                        wspace=.3, hspace=.2,
                                        margin_top=0.5,
                                        margin_bottom=0.5,
                                        margin_right=1.1,
                                        margin_left=1.1,
                                        )
flattened_axs = axs.flatten()

cmap = 'seismic'

for ax in axs.flatten():
    # If this comes after everything else, all subplots collapse for some reason
    ax.set_aspect('equal')  # forces 1:1 ratio per subplot
    # ax.legend(loc="best")

# Fill: Upper left plot
x_ul = Xi_ul/1e2
cb_1, im_1 = visualize_stress(stress_matrix=x_ul, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[0], delay_plot=True, colorbar_label="", return_aux=True,
                              cmap=cmap)
flattened_axs[0].set_title(r"White noise, unsmoothed")


# Fill: Upper right plot
x_ur = S.real/1e2
cb_2, im_2 = visualize_stress(stress_matrix=x_ur, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[1], delay_plot=True, colorbar_label=r"Stress $\mathrm{[10^{-2}]}$", return_aux=True, cmap=cmap)
flattened_axs[1].set_title(r"White noise Wigner, unsmoothed")


# Fill: Lower left plot
x_ll = Xi_ul
cb_3, im_3 = visualize_stress(stress_matrix=x_ll, rows=f, cols=t, smooth=True, custom_ax=flattened_axs[2], delay_plot=True, colorbar_label="", return_aux=True,
                               cmap=cmap)
flattened_axs[2].set_title(r"White noise, smoothed")


# Fill: Lower right plot
x_lr = smooth_matrix(S, smoothing_lvl=5).real
cb_4, im_4 = visualize_stress(stress_matrix=x_lr, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[3], delay_plot=True, return_aux=True,  cmap=cmap)
flattened_axs[3].set_title("White noise Wigner, smoothed")

# Make a bit more space for the colorbars
# fig.subplots_adjust(wspace=wspace, hspace=hspace, right=0.88)  # right: 1 => Axes extend until 100% of the widtH

# Adjust upper row colorbar limits: fix them to wigner limits
min_data_upper_rows = np.min(x_ur)
max_data_upper_rows = np.max(x_ur)

im_1.set_clim(min_data_upper_rows, max_data_upper_rows)
cb_1.update_normal(im_1)

# Adjust lower row colorbar limits: fix them to wigner limits (tested: cmap as well as values do change)
min_data_lower_rows = np.min(x_lr)
max_data_lower_rows = np.max(x_lr)
im_3.set_clim(min_data_lower_rows, max_data_lower_rows)
cb_3.update_normal(im_3)

flattened_axs[0].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[3].set_xlabel(r"Time $\mathrm{[s]}$")

# Finally append labels for back reference

labels = ["(a)", "(b)", "(c)", "(d)"]

for ax, lab in zip(flattened_axs, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.9, lw=0)

    )

# Manually adapt cbar limits if seismic
if cmap == 'seismic':
    # Update cbar levels manually
    im_1.set_clim(-1, 1)
    cb_1.update_normal(im_1)

    im_2.set_clim(-1, 1)
    cb_2.update_normal(im_2)

save_figure(save_fig=True, tight_ly=False)


print("\n")
print("Mean of white noise Wigner: ", np.mean(S).real, " should be ~ 1")
print("Std of white noise Wigner: ", np.std(S).real, f" should be ~ {np.round(np.sqrt(n_pix),2)}")

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


Mean of white noise Wigner:  1.0320226624982385  should be ~ 1
Std of white noise Wigner:  46.145645635008336  should be ~ 44.72


## Wigner of numerical relativity template and whitened data

In [30]:
# Get the data

# Whitened data of GW150914
GW150914 = get_strain_from_disc(add_whitened_data=True)
time_GW_event = GW150914.event_time
xi_GW_event = GW150914.event_strain_white
S_gw, t_gw, f_gw = Stress_jft(xi=xi_GW_event, time=time_GW_event, supress_print=True)

# Num rel data
nrt_strain_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt")
xi_nrt = nrt_strain_values
xi_time = nrt_time_values - nrt_time_values[0] - 1.393

S_nrt, t_nrt, f_nrt = Stress_jft(xi=xi_nrt, time=xi_time, supress_print=True)


	Constructing 15 windows over which we average.
	Mean variance of tapered windows: 4.542402182156302
	Compare with area under welch ps:  4.545029553571364


In [31]:
# Write helper function

def detection_statistic_dev(stress_matrix, raw=False, plot=True, time_var=None, custom_ax=None, title=None, normalize=False, lb=""):
    """

    :param stress_matrix: 2D array, shape (n_f, n_t):   Output from stress_jft() function in standard DFT order.
    :param raw: bool,                                   If True, the input matrix is not manipulated in any way.
    :param normalize: bool,                             If True, DC-line is divided by its max and the average is printed.

    Picks out the interference pattern on the DC-line of a Wigner-derived phase-space distribution through the following
    operations:

        1. Shifts from standard DFT order to DC-centered ordered along columns
        2. Smooths through Gaussian convolution
        3. Takes the absolute square. We call the resulting matrix 'smoothed Wigner power' (power => positive)
        4. Extracts DC-line from smoothed Wigner power

    :return dc_line,    The line corresponding to f=0 in the smoothed Wigner power
    :return SWP,        Smoothed Wigner power matrix in standard DFT order.

    """

    X = np.fft.fftshift(stress_matrix, axes=0)  # shift rows, corresponding to frequencies, such that f=0 is "in the middle" of the matrix. stress_matrix MUST be in standard DFT order (f=0 at the very bottom/top)

    if raw:
        SWP = X
    else:
        SWP = smooth_matrix(X, smoothing_lvl=5).real**2

    where_dc = SWP.shape[0]//2
    dc_line = SWP[where_dc, :]

    if normalize:
        dc_line /= np.max(dc_line)
        print("Average of smoothed Wigner power's DC-line: ", np.average(dc_line))

    if plot:
        if time_var is None:
            raise ValueError("To plot, please provide time array")
        if custom_ax is None:
            _ = plt.figure()
            axis = plt.gca()
        else:
            axis = custom_ax
        axis.plot(time_var, dc_line, label=lb, color="black")
    return dc_line, np.fft.ifftshift(SWP, axes=0)

In [51]:
# Build the sceleton

fig, axs = create_square_subplots_fixed(nrows=3, ncols=2,
                                  w_sub=4, # desired subplot width in inches
                                  h_sub=4, # desired subplot height in inches
                                  wspace=0.3, # vertical space between subplots as a fraction of total figsize
                                  hspace=0.2, # vertical space between subplots as a fraction of total figsize
                                  margin_top=0.5,
                                  margin_bottom=0.5,
                                  margin_right=1.1,
                                  margin_left=1.1,
                                  sharex=False, sharey=False)

flattened_axs = axs.flatten()

cmap = 'seismic'
for ax in axs.flatten():
    # If this comes after everything else, all subplots collapse for some reason
    ax.set_aspect('equal')  # forces 1:1 ratio per subplot

# Fill: Upper left plot
x_ul = S_nrt.real/np.max(S_nrt.real)
cb_1, im_1 = visualize_stress(stress_matrix=x_ul, rows=f_nrt, cols=t_nrt, smooth=False, custom_ax=flattened_axs[0], delay_plot=True, colorbar_label="", return_aux=True, cmap=cmap)
# flattened_axs[0].set_title(r"$S_{ft}$ of NRT, unsmoothed")
flattened_axs[0].set_title(r"Numerical relativity")


# Fill: Upper right plot
x_ur = S_gw.real/np.max(S_gw.real)
cb_2, im_2 = visualize_stress(stress_matrix=x_ur, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[1], delay_plot=True, colorbar_label=r"Stress (arb. units)", return_aux=True, cmap=cmap)
# flattened_axs[1].set_title(r"$S_{ft}$ of whitened GW150914 data, unsmoothed")
flattened_axs[1].set_title(r"Whitened data")


# Fill: Lower left plot
x_ll = smooth_matrix(S_gw.real,5)
x_ll /= np.max(x_ll)
cb_3, im_3 = visualize_stress(stress_matrix=x_ll, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[2], delay_plot=True, colorbar_label="", return_aux=True, cmap=cmap)
# flattened_axs[2].set_title(r"$S_{ft}$ of whitened GW150914 data, smoothed")
flattened_axs[2].set_title(r"As (b) but smoothed")


# Fill: Lower right plot
x_lr = smooth_matrix(S_gw.real, 5)**2
x_lr /= np.max(x_lr)
cb_4, im_4 = visualize_stress(stress_matrix=x_lr, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[3], delay_plot=True, return_aux=True, colorbar_label="Stress (arb. units)", cmap='Reds')
flattened_axs[3].set_title(r"Square of (c)")


# Fill plot under lower right plot (lower-lower right)
x_llr = x_ll
x_llr /= np.max(x_llr)
_ = detection_statistic_dev(stress_matrix=x_llr, time_var=t_gw, custom_ax=flattened_axs[5])
flattened_axs[5].set_title(r"$f=0$ (DC) line of (d)")
flattened_axs[5].set_aspect('auto')  # otherwise collapses because set_aspect is globally set to 'equal' but this plot has different axes ranges

# Get rid of unneccessary axis
flattened_axs[4].set_axis_off()

# Make a bit more space for the colorbars
# fig.subplots_adjust(wspace=wspace, hspace=hspace, right=0.88, top=0.95, bottom=0.05)  # right: 1 => Axes extend until 100% of the width

# Adjust upper row colorbar limits: fix them to wigner limits
min_data = np.min(x_ul)
max_data = np.max(x_ul)

im_2.set_clim(min_data, max_data)
im_3.set_clim(min_data, max_data)
cb_2.update_normal(im_1)
cb_3.update_normal(im_1)


flattened_axs[0].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[5].set_ylabel(r"$(S \ast G)^2\vert_{f=0}$")

# flattened_axs[3].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[2].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[5].set_xlabel(r"Time $\mathrm{[s]}$")

flattened_axs[1].set_yticklabels([])
flattened_axs[3].set_yticklabels([])

# Set x and y limits
independent_ax = flattened_axs[5]
for ax in flattened_axs:
    if ax is not independent_ax:
        ax.set_xlim(-.14,.1)
        ax.set_ylim(-350,350)
    else:
        ax.set_xlim(-.14,.1)

# Finally append labels for back reference

labels = ["(a)", "(b)", "(c)", "(d)", "", "(e)"]

for ax, lab in zip(flattened_axs, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.6, lw=0)

    )

save_figure(True, tight_ly=False)


		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Detection statistic over full domain

In [71]:
# Get the data
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

# Whitened data of GW150914
GW150914_tmp = get_strain_from_disc(add_whitened_data=True, event_name="GW150914", detector="H1", data_duration="32sec")
time_GW_event_tmp = GW150914_tmp.event_time
xi_GW_event_tmp = GW150914_tmp.event_strain_white
S_gw_tmp, t_gw_tmp, f_gw_tmp = Stress_jft(xi=xi_GW_event_tmp, time=time_GW_event_tmp, supress_print=True)

# plt.plot(GW150914_tmp.event_time, GW150914_tmp.event_strain_white)
# plt.show()
visualize_stress(S_gw_tmp,f_gw_tmp, t_gw_tmp, smooth=True)

_ = detection_statistic_dev(S_gw_tmp, time_var=t_gw_tmp)


Start: Calculating welch average

Constructing 16 windows over which we average.

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
time and xi lengths:  4097 4097


## Detection statistic for smooth-zeropaded Wigner function (developmental)

In [33]:
# Define necessary functions
import jax.numpy as jnp

def boundary_differences(M):
    """
    M: 2D array (can be complex)
    returns:
        col_diff, row_diff  (complex arrays)
    """
    M = np.asarray(M)

    # columns: last row - first row
    col_diff = M[-1, :] - M[0, :]

    # rows: last column - first column
    row_diff = M[:, -1] - M[:, 0]

    return col_diff, row_diff

def plot_boundary_differences(M):
    col_diff, row_diff = boundary_differences(M)

    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=False)

    # columns
    axs[0].plot(col_diff.real, label="Re(col diff)")
    axs[0].plot(col_diff.imag, label="Im(col diff)")
    axs[0].set_title("Column boundary differences")
    axs[0].set_xlabel("Column index")
    axs[0].legend()

    # rows
    axs[1].plot(row_diff.real, label="Re(row diff)")
    axs[1].plot(row_diff.imag, label="Im(row diff)")
    axs[1].set_title("Row boundary differences")
    axs[1].set_xlabel("Row index")
    axs[1].legend()

    plt.tight_layout()
    plt.show()

def pad_matrix(M, extent=0.1):
    """
    Pad a square matrix on all sides with its boundary values.

    Parameters
    ----------
    M : 2D square array
    extent : float
        Fraction of matrix size to pad on each side

    Returns
    -------
    M_padded : 2D array
        Padded matrix
    """
    M = np.asarray(M)
    if M.shape[0] != M.shape[1]:
        raise ValueError("M must be square")

    n = M.shape[0]
    pad = int(n * extent)

    print("Padding with ", pad, " elements, corresponding to ", np.round(100*pad/n,2), "% of array length")

    # top/bottom padding: repeat first/last row
    top = np.repeat(M[0:1, :], pad, axis=0)
    bottom = np.repeat(M[-1:, :], pad, axis=0)

    M_vert = np.vstack([top, M, bottom])

    # left/right padding: repeat first/last column
    left = np.repeat(M_vert[:, 0:1], pad, axis=1)
    right = np.repeat(M_vert[:, -1:], pad, axis=1)

    M_padded = np.hstack([left, M_vert, right])
    return M_padded

def smooth_zero_pad_core(M, extent=0.1, alpha=1, beta=.5):
    """
    Pad a square matrix with its boundary values and taper only the padding toward zero.
    Inner core remains untouched.

    Parameters
    ----------
    M : 2D square array
    extent : float
        Fraction of matrix size used as padding

    Returns
    -------
    M_smooth : 2D array
        Padded matrix with aggressively tapered edges
    """
    M = np.asarray(M)
    n = M.shape[0]
    pad = int(n * extent)
    if pad == 0:
        return M.copy()

    # Step 1: pad with boundary values
    M_pad = pad_matrix(M, extent=extent)
    N = M_pad.shape[0]

    # Step 2: create taper windows for padding region only
    wx = np.ones(N)
    wy = np.ones(N)

    # Left/right taper
    if pad > 0:
        wx[:pad] = (beta * (1 - np.cos(np.pi * np.linspace(0, 1, pad))))**alpha
        wx[-pad:] = (beta * (1 - np.cos(np.pi * np.linspace(1, 0, pad))))**alpha
        wy[:pad] = (beta * (1 - np.cos(np.pi * np.linspace(0, 1, pad))))**alpha
        wy[-pad:] = (beta * (1 - np.cos(np.pi * np.linspace(1, 0, pad))))**alpha

    # Step 3: make 2D window
    window2d = wy[:, None] * wx[None, :]

    # Step 4: preserve inner core
    inner_slice = slice(pad, N-pad)
    M_smooth = M_pad.copy()
    # multiply only padding regions
    # top
    M_smooth[:pad, :] *= window2d[:pad, :]
    # bottom
    M_smooth[-pad:, :] *= window2d[-pad:, :]
    # left
    M_smooth[pad:-pad, :pad] *= window2d[pad:-pad, :pad]
    # right
    M_smooth[pad:-pad, -pad:] *= window2d[pad:-pad, -pad:]

    col_diff, row_diff = boundary_differences(M_smooth)

    if np.any(np.round(np.sum(col_diff.real), 12) != 0) or \
       np.any(np.round(np.sum(col_diff.imag), 12) != 0) or \
       np.any(np.round(np.sum(row_diff.real), 12) != 0) or \
       np.any(np.round(np.sum(row_diff.imag), 12) != 0):
        raise ValueError("Boundary differences are nonzero; increase extent to reduce leakage.")

    return M_smooth

def Stress_re_debug_custom_pad(xi, time, padding_extent=.1, supress_print=False, downsample=False, norm="ortho", tukey_window_where_necessary=False,
                               zp_func=smooth_zero_pad_core):
    """
    Implements S_ft, i.e. rows are frequencies and columns are times.

    See also nifty8 `Stress` function.

    :param xi: jnp.array        A field to calculate the wigner function for. Either of complex or real data type.
                                If complex, assumed to be in DFT standard order (DC first, then positives then negatives).
    :param time: jnp.array      The real-space time array at which xi (or its iFFT if complex) was sampled at.
    :param supress_print: bool, Print imaginary part diagonstics (Wigner function should be real).
    :return:
    """

    t0 = time[0]
    dt = time[1]-time[0]

    # extent = 1 + padding_extent
    N = len(xi) # * extent
    f = jnp.fft.fftfreq(N, d=dt)
    k = f.copy()
    df = f[1] - f[0]
    t = jnp.arange(N) / (N*df)  # dual time, equal to input time - time[0].
    T = N * dt

    FFT_physical = lambda x, ax=-1: jnp.fft.fft(x, norm=norm, axis=ax) * T / jnp.sqrt(N)
    iFFT_physical = lambda x, ax=-1: jnp.fft.ifft(x, norm=norm, axis=ax) * jnp.sqrt(N) / T

    if jnp.iscomplexobj(xi):
        print("INVERSE FOURIER TRANSFORMING xi")
        xi = iFFT_physical(xi)  # go to real space
    else:
        print("not INVERSE FOURIER TRANSFORMING xi")

    if downsample:
        step = 2
        xi = xi[::step]
        time = time[::step]

    if not supress_print:
        print("\nCalculating stress...")

    t_c = t[:, None]  # time cast
    k_c = k[None, :]  # shift frequencies cast
    xi_c = xi[:, None]  # xi values cast as rows

    if not supress_print:
        print("\t Calculating zeta plus")
    if tukey_window_where_necessary:
        # plot_boundary_differences(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c)
        # zeta_plus = tukey_window_matrix(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        zeta_plus = zp_func(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, padding_extent) # domain = (time_space, h_space)
        plot_boundary_differences(zeta_plus)
    else:
        zeta_plus = jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta minus")

    if tukey_window_where_necessary:
        # plot_boundary_differences(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c)
        # zeta_minus = tukey_window_matrix(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
        zeta_minus = zp_func(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, padding_extent) # domain = (time_space, h_space)
        plot_boundary_differences(zeta_minus)
        # stop
    else:
        zeta_minus = jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c  # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT_physical(zeta_plus, ax=0)

    if not supress_print:
        print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT_physical(zeta_minus, ax=0)

    if not supress_print:
        print("\t Calculating Phi matrix")
    if tukey_window_where_necessary:
        # plot_boundary_differences(tilde_zeta_plus * tilde_zeta_minus.conj())
        # Phi = tukey_window_matrix(tilde_zeta_plus * tilde_zeta_minus.conj(), ax=1)  # domain = (h_space, h_space)
        print("NOT TUKEYING PHI, since it looks like in all cases its almost periodic...")
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)
        plot_boundary_differences(Phi)
    else:
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)

    if not supress_print:
        print("\t Inverse Fourier-Transforming columns of Phi matrix")
    S = iFFT_physical(Phi, ax=1)
    S.block_until_ready()

    if not supress_print:
        print("\t ... Done")
    if not supress_print:
        diagnostic = jnp.abs(jnp.mean(S.imag))
        tmp = float(diagnostic)
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
        else:
            raise_warning(
                f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")

    if tukey_window_where_necessary:
        pad = int(len(xi) * padding_extent)
        N_pad = len(xi) + 2*pad
        f_pad = jnp.fft.fftfreq(N_pad, d=dt)
        df_pad = f[1] - f[0]
        t_pad = jnp.arange(N_pad) / (N_pad*df_pad)  # dual time, equal to input time - time[0].
        return S, t_pad+t0, f_pad
    else:
        return S, t+t0, f


In [34]:
# Get data
S_mat_nrt_padded, t_dual_nrt_padded, f_dual_nrt_padded = Stress_re_debug_custom_pad(xi=nrt_strain_values, time=xi_time, tukey_window_where_necessary=True,
                                                                                    padding_extent=0.01, zp_func=smooth_zero_pad_core)
S_mat_white_padded, t_dual_white_padded, f_dual_white_padded = Stress_re_debug_custom_pad(xi=np.random.standard_normal(len(nrt_time_values)), time=xi_time, tukey_window_where_necessary=True,
                                                                               padding_extent=0.01, zp_func=smooth_zero_pad_core)

not INVERSE FOURIER TRANSFORMING xi

Calculating stress...
	 Calculating zeta plus
Padding with  81  elements, corresponding to  0.99 % of array length
	 Calculating zeta minus
Padding with  81  elements, corresponding to  0.99 % of array length
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
NOT TUKEYING PHI, since it looks like in all cases its almost periodic...
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (4.65949252583357e-26) 
not INVERSE FOURIER TRANSFORMING xi

Calculating stress...
	 Calculating zeta plus
Padding with  81  elements, corresponding to  0.99 % of array length
	 Calculating zeta minus
Padding with  81  elements, corresponding to  0.99 % of array length
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
NOT TUKEYING PHI, since it looks like in all cases its almos

In [46]:
# Plot
fig, axs = create_square_subplots_fixed(nrows=1, ncols=2,
                                  w_sub=4, # desired subplot width in inches
                                  h_sub=4, # desired subplot height in inches
                                  wspace=0.3, # vertical space between subplots as a fraction of total figsize
                                  hspace=0.2, # vertical space between subplots as a fraction of total figsize
                                  margin_top=0.5,
                                  margin_bottom=0.5,
                                  margin_right=1.3,
                                  margin_left=1.1,
                                  sharex=True, sharey=True)

for ax in axs:
    # If this comes after everything else, all subplots collapse for some reason
    ax.set_aspect('equal')  # forces 1:1 ratio per subplot

cmap = 'seismic'

# Fill left plot
x_l = S_mat_nrt_padded
x_l /= np.max(x_l)
cb_1, im_1 = visualize_stress(stress_matrix=x_l, rows=f_dual_nrt_padded, cols=t_dual_nrt_padded, smooth=False, custom_ax=axs[0], delay_plot=True, colorbar_label=r"", return_aux=True, cmap=cmap)
axs[0].set_title(r"Numerical relativity")

# Fill right plot
x_r = smooth_matrix(S_mat_white_padded, 5)
x_r /= np.max(x_r)
cb_2, im_2 = visualize_stress(stress_matrix=x_r, rows=f_dual_white_padded, cols=t_dual_white_padded, smooth=False, custom_ax=axs[1], delay_plot=True, colorbar_label=r"Stress (arb. units)", return_aux=True, cmap=cmap)
axs[1].set_title(r"White noise Wigner, smoothed")

# Set axes labels and limits
axs[0].set_xlabel(r"Time $\mathrm{[s]}$")
axs[1].set_xlabel(r"Time $\mathrm{[s]}$")
axs[0].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
axs[1].set_ylabel(r"")
axs[0].set_xlim(-0.26, 0.145)
axs[0].set_ylim(-400, 400)

# Update cbar levels manually
im_2.set_clim(-0.1, 0.1)
cb_2.update_normal(im_2)

save_figure(True, tight_ly=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Detection statistic over whole domain with matrix-padding scheme

In [8]:
# Get the data
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

# Whitened data of GW150914
GW150914_tmp = get_strain_from_disc(add_whitened_data=True, event_name="GW150914", detector="H1", data_duration="32sec")
time_GW_event_tmp = GW150914_tmp.event_time
xi_GW_event_tmp = GW150914_tmp.event_strain_white
# S_gw_tmp, t_gw_tmp, f_gw_tmp = Stress_jft(xi=xi_GW_event_tmp, time=time_GW_event_tmp, supress_print=True)
S_gw_tmp, t_gw_tmp, f_gw_tmp = Stress_re_debug_custom_pad(xi=xi_GW_event_tmp, time=time_GW_event_tmp, supress_print=True,
                                                          tukey_window_where_necessary=True, padding_extent=0.01, zp_func=smooth_zero_pad_core)

# plt.plot(GW150914_tmp.event_time, GW150914_tmp.event_strain_white)
# plt.show()
visualize_stress(S_gw_tmp,f_gw_tmp, t_gw_tmp, smooth=True)

_ = detection_statistic_dev(S_gw_tmp, time_var=t_gw_tmp)


Start: Calculating welch average

Constructing 16 windows over which we average.

not INVERSE FOURIER TRANSFORMING xi
Padding with  81  elements, corresponding to  0.99 % of array length
Padding with  81  elements, corresponding to  0.99 % of array length
NOT TUKEYING PHI, since it looks like in all cases its almost periodic...
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
